In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model='gpt-4o-mini')


In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz


@tool
def get_current_time(timezone:str, location:str) -> str:
    """
    현재 시각을 반환하는 함수
    
    Args:
        timezone(str) : 타임존(예: 'Asia/Seoul'). 실제 존재해야함.
        location(str) : 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨.
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각 {now}'
    print(location_and_local_time)
    return location_and_local_time

In [3]:
tools = [get_current_time]
tool_dict = {"get_current_time":get_current_time}

llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 129, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ff247d5857', 'id': 'chatcmpl-DbQTXNsD6gK0zWdlPlfgPMKtOQ7DK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dedde-8fa6-78e2-8be3-5aa1f8197458-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'}, 'id': 'call_ZLNBCGRbaamdOpV9fOwQQDJF', 'type': 'tool_call'

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call['name']]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': 'Busan'}
Asia/Seoul (Busan) 현재 시각 2026-05-03 21:44:40


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 129, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ff247d5857', 'id': 'chatcmpl-DbQTXNsD6gK0zWdlPlfgPMKtOQ7DK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dedde-8fa6-78e2-8be3-5aa1f8197458-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'}, 'id': 'call_ZLNBCGRbaamdOpV9fOwQQDJF', 'type': 'tool_cal

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='현재 부산의 시각은 2026년 5월 3일 21시 44분 40초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 185, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ff247d5857', 'id': 'chatcmpl-DbQTZtWurrza69OH0MGKllz5JWa0d', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dedde-9651-7863-84eb-e99b159515ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'output_tokens': 28, 'total_tokens': 213, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field



class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")
    

In [8]:
import yfinance as yf


@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history.md = history.to_markdown()

    return history.md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time":get_current_time, "get_yf_stock_history":get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("구글 주가는 한달사이 어떻게됬어?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 270, 'total_tokens': 298, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_97c29773b5', 'id': 'chatcmpl-DbQTbyK4R9uKx9jFIbZNiGF1FXTne', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019dedde-9d91-79a0-9310-eaa670349c9a-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'GOOGL', 'period': '1mo'}}, 'id': 'call_icvQqTKydvB9QbR0DIMVysUR', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 270, 'output_tokens': 28, 'total_tokens': 298, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'aud

In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'GOOGL', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-04-02 00:00:00-04:00 | 290.69 | 298.08 | 289.45 |  295.77 | 2.16665e+07 |           0 |              0 |\n| 2026-04-06 00:00:00-04:00 | 295.87 | 300.62 | 295.18 |  299.99 | 1.69455e+07 |           0 |              0 |\n| 2026-04-07 00:00:00-04:00 | 302.73 | 305.63 | 297.72 |  305.46 | 2.32054e+07 |           0 |              0 |\n| 2026-04-08 00:00:00-04:00 | 320.45 | 322.08 | 315.02 |  317.32 | 3.35471e+07 |           0 |              0 |\n| 2026-04-09 00:00:00-04:00 | 315.91 | 319.54 | 311.06 |  318.49 | 2.37392e+07 |           0 |              0 |\n| 2026-04-10 00:00:00-04:00 | 320.02 | 321.83 | 316.32 |  317.24 | 1.91526e+07 |           0 |              0 |\n| 2026-04-13 00:00:00-04:

In [11]:
llm_with_tools.invoke(messages)

AIMessage(content='구글(Alphabet Inc.)의 주가는 최근 한 달 동안 다음과 같이 변동하였습니다:\n\n- **2026-04-02**: 개장가 290.69 → 종가 295.77\n- **2026-04-06**: 개장가 295.87 → 종가 299.99\n- **2026-04-07**: 개장가 302.73 → 종가 305.46\n- **2026-04-08**: 개장가 320.45 → 종가 317.32\n- **2026-04-09**: 개장가 315.91 → 종가 318.49\n- **2026-04-10**: 개장가 320.02 → 종가 317.24\n- **2026-04-13**: 개장가 317.14 → 종가 321.31\n- **2026-04-14**: 개장가 324.79 → 종가 332.91\n- **2026-04-15**: 개장가 332.89 → 종가 337.12\n- **2026-04-16**: 개장가 338.75 → 종가 336.02\n- **2026-04-17**: 개장가 337.65 → 종가 341.68\n- **2026-04-20**: 개장가 340.76 → 종가 337.42\n- **2026-04-21**: 개장가 337.69 → 종가 332.29\n- **2026-04-22**: 개장가 337.02 → 종가 339.32\n- **2026-04-23**: 개장가 341.18 → 종가 338.89\n- **2026-04-24**: 개장가 338.73 → 종가 344.40\n- **2026-04-27**: 개장가 345.98 → 종가 350.34\n- **2026-04-28**: 개장가 348.55 → 종가 349.78\n- **2026-04-29**: 개장가 347.57 → 종가 349.94\n- **2026-04-30**: 개장가 374.07 → 종가 384.80\n- **2026-05-01**: 개장가 381.63 → 종가 385.69\n\n**결론적으로:** 한 달 전인 2026년 4월 2일의 종가 295.77에서 현재

In [12]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점이 무엇인지 이야기해줘.")]):
    print(c.content, end="|")

|안|녕하세요|!| 한국| 사회|에는| 다양한| 문제|점|들이| 존재|합니다|.| 몇| 가지| 주요| 문제|를| 소개|해|드|릴|게|요|.

|1|.| **|경제|적| 양|극|화|**|:| 한국|은| 고|소|득|층|과| 저|소|득|층| 간|의| 격|차|가| 점|점| 더| 벌|어|지고| 있습니다|.| 중|산|층|의| 감소|와| 청|년|실|업| 문제가| 심|각|하게| 나타|나|고| 있으며|,| 이|로| 인해| 사회|적| 불|만|이| 고|조|되고| 있습니다|.

|2|.| **|주|거| 문제|**|:| 부|동|산| 가격| 상승|과| 전|세| 제|도의| 불|안|정|함|으로| 인해| 젊|은| 세|대|가| 주|거| 문제|를| 겪|고| 있습니다|.| 많은| 청|년|들이| 안정|적인| 주|거| 환경|을| 찾|기| 어려|워|하고| 있습니다|.

|3|.| **|고|령|화| 사회|**|:| 인|구| 고|령|화|는| 한국| 사회|에| 큰| 도|전| 과|제가| 되고| 있습니다|.| 노|인| 인|구|의| 증가|로| 인해| 건강| 관리|,| 연|금| 시스템|,| 사회| 복|지| 등| 다양한| 분야|에서| 부담|이| 커|지고| 있습니다|.

|4|.| **|성| 불|평|등|**|:| 성|별|에| 따른| 불|평|등| 문제|도| 여|전히| 해결|되지| 않고| 있습니다|.| 직|장에서|의| 성|차|별|,| 성|폭|력| 문제| 등|은| 사회|적으로| 큰| 이|슈|로| 다|뤄|지고| 있습니다|.

|5|.| **|정|신| 건강|**|:| 정신| 건강| 문제가| 사회|적으로| 크게| 주|목|받|고| 있지만|,| 여|전히| stig|mat|ization|이| 존재|하여| 많은| 이|들이| 치료|를| 받|기| 어려|운| 상황|입니다|.

|6|.| **|교육| 경쟁|**|:| 치|열|한| 교육| 경쟁|이| 학생|들에게| 극|심|한| 스트|레|스를| 부|여|하고| 있으며|,| 이는| 정신|적인| 문제|로| 이어|질| 수| 있습니다|.| 또한| 사|교육|비|의| 부담|이| 많은| 부모|들에게| 경제|적인

In [13]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?")
]

response = llm_with_tools.stream(messages)

is_first = True

for chunk in response:
    print("chunk type:: ", type(chunk))
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type::  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}]
chunk type::  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}]
chunk type::  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}]
chunk type::  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}]
chunk type::  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 't

In [14]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_97c29773b5', 'service_tier': 'default'}, id='lc_run--019dede4-726c-7d02-806e-160b20b505ef', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_tokens': 23, 'total_tokens': 220, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [15]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_call)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-05-03 21:52:26


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_97c29773b5', 'service_tier': 'default'}, id='lc_run--019dede4-726c-7d02-806e-160b20b505ef', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_tokens': 23, 'total_tokens': 220, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'index': 0, 'ty

In [16]:
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")

ValueError: Message dict must contain 'role' and 'content' keys, got {'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_oxo0KIw4Rd52mILpf7tmSXCK', 'type': 'tool_call'}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_COERCION_FAILURE 